In [1]:
import pandas as pd
import nltk
import string
import random
import pickle

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.classify import NaiveBayesClassifier
from nltk.classify.util import accuracy

CARGA DE DATOS DE ENTRENAMIENTO

In [2]:
df = pd.read_excel(r"Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\Hogar\analisis_hogar.xlsx")

CREAR UN NUEVO DATAFRAME A PARTIR DE ESTE QUE CONTENGA UNICAMENTE EL TEXTO, LA ETIQUETA Y EL IDENTIFICADOR

In [3]:
columnas_necesarias = [
    "texto_norm",
    "sentimiento",
    "confianza"
]
df = df[columnas_necesarias]

import re
def texto_valido(texto):
    if not isinstance(texto, str):
        return False
    texto = texto.strip()
    if texto == "":
        return False
    if len(texto) < 3:
        return False
    # comprobar que tenga al menos una letra
    if not re.search(r"[a-zA-ZáéíóúÁÉÍÓÚñÑ]", texto):
        return False
    return True

df = df[df["texto_norm"].apply(texto_valido)]

df["confianza"] = pd.to_numeric(df["confianza"], errors="coerce")
df = df[df["confianza"] >= 0.95]

print(df.head)
print(df.shape)

<bound method NDFrame.head of                                                texto_norm sentimiento  \
0       hoy a 1 de enero de 2026 ningun operació ha ve...    Negativo   
1          la atencion recibida , y la buena resolucion .    Positivo   
3       se trataba de un problema eléctrico que hacía ...    Positivo   
4       la perito me explico los pasosy ls valoravion ...    Positivo   
5                            las 4 reflejan mi valoración    Positivo   
...                                                   ...         ...   
227838                                eficaz y muy amable    Positivo   
227839                                   buen profesional    Positivo   
227840             puntualidad y explicacion del problema    Positivo   
227841   por esta vía me abstengo a detallar lo ocurrido.    Positivo   
227842  la persona que vino no ha solucionado el probl...    Negativo   

        confianza  
0        0.997983  
1        0.994089  
3        0.967096  
4        0.99

PRPROCESAMIENTO DEL TEXTO

In [5]:
stop_words = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    texto = texto.lower()
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(texto)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

df["tokens_limpios"] = df["texto_norm"].apply(limpiar_texto)
df = df[df["tokens_limpios"].str.len() > 0]

ENTRENAMIENTO

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# X e y
X = df["texto_norm"]   
y = df["sentimiento"]   

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),   # añade trigramas
    min_df=3,             # más vocabulario
    max_df=0.85,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Logistic Regression
model = LogisticRegression(
    max_iter=2000,
    C=2.0,
    class_weight="balanced",
    solver="liblinear"
)

model.fit(X_train_tfidf, y_train)

# Evaluación
y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9083221701272606


VALIDACION CRUZADA

In [9]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    X_train_tfidf,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Accuracy por fold:", scores)
print("Accuracy medio CV:", scores.mean())
print("Desviación:", scores.std())

Accuracy por fold: [0.90797139 0.90908774 0.90612245 0.90765742 0.90751465]
Accuracy medio CV: 0.907670730958678
Desviación: 0.0009507913292093963


GUARDAR MODELO

In [10]:
import joblib

joblib.dump(model, "modelo_sentimiento_hogar_logreg.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer_hogar.pkl")

['tfidf_vectorizer_hogar.pkl']

CARGAR MODELO

In [ ]:
model = joblib.load("modelo_sentimiento_logreg.pkl")
vectoriZer = joblib.load("tfidf_vectorizer.pkl")